# PRJ-022: train the answer classifier

Teaches a small language model to say whether a **student answer** *supports*, *contradicts*,
or does *neither* for a **model answer**.

**Colab:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**.
The last cell downloads `prj022_model.zip`.
**Kaggle:** Settings > Accelerator > **GPU**, Internet **on**, then **Run all**.
Get `prj022_model.zip` from the Output panel.

Then put `prj022_model.zip` into the project's `model` folder and restart `start_server.bat`.

In [ ]:
!pip -q install datasets sentencepiece

## 1. Settings
To compare, change `MODEL` to `"distilbert-base-uncased"` (no entailment knowledge, so give it `EPOCHS = 4`) and run all again.

In [ ]:
MODEL = "cross-encoder/nli-distilroberta-base"   # small and fast: 6 layers
EPOCHS = 2

## 2. Biology pairs written for this project
`S` = supports, `C` = contradicts, `N` = neither. Add your own rows to fit your syllabus.

In [ ]:
BIOLOGY_TRAIN = [
 ("Mitochondria are where the cell gets its energy from respiration.", "Mitochondria release energy through aerobic respiration.", "S"),
 ("The nucleus holds the DNA and controls the cell.", "The nucleus contains genetic material and controls the activities of the cell.", "S"),
 ("Ribosomes make proteins.", "Protein synthesis takes place at the ribosomes.", "S"),
 ("Chloroplasts trap light energy to make glucose.", "Chloroplasts absorb light energy for photosynthesis.", "S"),
 ("Osmosis is water moving through a partially permeable membrane from a dilute solution to a concentrated one.", "Osmosis is the movement of water from a dilute solution to a more concentrated solution through a partially permeable membrane.", "S"),
 ("Enzymes speed up reactions in the body.", "Enzymes are biological catalysts that increase the rate of reactions.", "S"),
 ("The substrate fits into the active site like a key in a lock.", "The substrate has a shape that fits the active site of the enzyme.", "S"),
 ("Red blood cells carry oxygen around the body.", "Red blood cells transport oxygen.", "S"),
 ("Veins have valves to stop the blood going backwards.", "Valves in veins prevent the backflow of blood.", "S"),
 ("Xylem carries water up from the roots.", "Xylem vessels transport water from the roots to the leaves.", "S"),
 ("Mitosis makes two cells that are genetically the same.", "Mitosis produces two genetically identical cells.", "S"),
 ("Mitochondria are where photosynthesis happens.", "Photosynthesis takes place in the chloroplasts.", "C"),
 ("Animal cells have a cell wall.", "Animal cells do not have a cell wall.", "C"),
 ("Osmosis moves water from a concentrated solution to a dilute solution.", "Osmosis is the movement of water from a dilute solution to a more concentrated solution.", "C"),
 ("Diffusion needs energy from respiration.", "Diffusion is a passive process that does not need energy.", "C"),
 ("Enzymes are used up in the reaction.", "Enzymes are not used up in the reactions they catalyse.", "C"),
 ("Enzymes work faster the hotter it gets, with no limit.", "Above the optimum temperature the enzyme is denatured and the rate falls.", "C"),
 ("Arteries carry blood towards the heart.", "Arteries carry blood away from the heart.", "C"),
 ("Plants take in oxygen to make food in photosynthesis.", "Plants take in carbon dioxide for photosynthesis.", "C"),
 ("Anaerobic respiration uses oxygen.", "Anaerobic respiration takes place without oxygen.", "C"),
 ("Phloem carries water and xylem carries sugar.", "Xylem transports water and phloem transports sugars.", "C"),
 ("Antibiotics kill viruses.", "Antibiotics kill bacteria but have no effect on viruses.", "C"),
 ("Insulin raises the blood glucose level.", "Insulin lowers the blood glucose concentration.", "C"),
 ("The cell shrinks because water leaves it.", "The cell swells because water enters it by osmosis.", "C"),
 ("Mitochondria are found in the cytoplasm.", "Mitochondria release energy through aerobic respiration.", "N"),
 ("Plants are green.", "Chloroplasts absorb light energy for photosynthesis.", "N"),
 ("Enzymes are proteins.", "Enzymes are denatured at high temperatures.", "N"),
 ("The heart is in the chest.", "The heart pumps blood around the body.", "N"),
 ("Water is important for cells.", "Osmosis is the movement of water from a dilute solution to a more concentrated solution.", "N"),
 ("I don't know.", "Xylem vessels transport water from the roots to the leaves.", "N"),
 ("Because it is hot.", "Above the optimum temperature the enzyme is denatured and the rate falls.", "N"),
 ("Bacteria can cause disease.", "Antibiotics kill bacteria but have no effect on viruses.", "N"),
 ("Arteries are blood vessels.", "Arteries carry blood away from the heart.", "N"),
 ("Glucose is a sugar.", "Insulin lowers the blood glucose concentration.", "N"),
 ("The cell changes.", "The cell swells because water enters it by osmosis.", "N"),
]

# Never trained on: used only to check the model on biology at the end.
BIOLOGY_TEST = [
 ("Alveoli have a big surface area so gases diffuse quickly.", "The alveoli provide a large surface area for gas exchange.", "S"),
 ("Root hair cells absorb water and minerals from the soil.", "Root hair cells take up water and mineral ions from the soil.", "S"),
 ("Platelets help the blood to clot.", "Platelets are involved in blood clotting.", "S"),
 ("Meiosis makes gametes with half the number of chromosomes.", "Meiosis produces gametes with half the chromosome number.", "S"),
 ("Oxygen moves from the blood into the alveoli.", "Oxygen diffuses from the alveoli into the blood.", "C"),
 ("Meiosis makes identical body cells.", "Meiosis produces genetically different gametes.", "C"),
 ("The pulmonary artery carries oxygenated blood.", "The pulmonary artery carries deoxygenated blood to the lungs.", "C"),
 ("Producers get their energy by eating other organisms.", "Producers make their own food by photosynthesis.", "C"),
 ("The lungs are in the chest.", "The alveoli provide a large surface area for gas exchange.", "N"),
 ("Roots are underground.", "Root hair cells take up water and mineral ions from the soil.", "N"),
 ("Blood has plasma in it.", "Platelets are involved in blood clotting.", "N"),
 ("Gametes are sex cells.", "Meiosis produces gametes with half the chromosome number.", "N"),
]

## 3. Load the data
SciEntsBank and Beetle (SemEval-2013 Task 7): real student science answers, each marked by teachers against a reference answer.

In [ ]:
import shutil, sys
import pandas as pd
import torch
from datasets import load_dataset

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow)")

LETTER = {"S": "supports", "C": "contradicts", "N": "neither"}


def from_hub(split):
    names = split.features["label"].names
    # teacher labels -> verdicts: correct = supports, contradictory = contradicts, rest = neither
    verdict = {"correct": "supports", "contradictory": "contradicts"}
    return pd.DataFrame({"student": list(split["student_answer"]),
                         "reference": list(split["reference_answer"]),
                         "verdict": [verdict.get(names[i], "neither") for i in split["label"]]})


def from_pairs(rows):
    return pd.DataFrame([(s, r, LETTER[v]) for s, r, v in rows],
                        columns=["student", "reference", "verdict"])


sci = load_dataset("nkazi/SciEntsBank")
beetle = load_dataset("nkazi/Beetle")
train_df = pd.concat([from_hub(sci["train"]), from_hub(beetle["train"])]
                     + [from_pairs(BIOLOGY_TRAIN)] * 3)      # biology counted 3 times
train_df = train_df.sample(frac=1, random_state=0)           # shuffle
tests = {"SciEntsBank, unseen questions": from_hub(sci["test_uq"]),
         "SciEntsBank, unseen science topics": from_hub(sci["test_ud"]),
         "Biology check": from_pairs(BIOLOGY_TEST)}
print(len(train_df), "training pairs")
print(train_df["verdict"].value_counts())

## 4. Train

In [ ]:
from transformers import (AutoConfig, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

# An entailment model already knows entailment / contradiction / neutral:
# keep its outputs and just rename them, so training starts from that knowledge.
config = AutoConfig.from_pretrained(MODEL)
names = [str(config.id2label[i]).lower() for i in range(len(config.id2label))]
RENAME = {"entailment": "supports", "contradiction": "contradicts", "neutral": "neither"}
if sorted(names) == sorted(RENAME):
    LABELS = [RENAME[n] for n in names]
else:
    LABELS = ["supports", "contradicts", "neither"]

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3, id2label=dict(enumerate(LABELS)),
    label2id={name: i for i, name in enumerate(LABELS)}, ignore_mismatched_sizes=True)


class Pairs(torch.utils.data.Dataset):
    """(student answer, model answer) pairs, ready for the model."""

    def __init__(self, df):
        self.enc = tokenizer(list(df["student"].astype(str)), list(df["reference"].astype(str)),
                             truncation=True, max_length=128)
        self.labels = [LABELS.index(v) for v in df["verdict"]]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


args = TrainingArguments(output_dir="runs", num_train_epochs=EPOCHS, learning_rate=2e-5,
                         per_device_train_batch_size=32, save_strategy="no",
                         logging_steps=100, report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=Pairs(train_df),
                  data_collator=DataCollatorWithPadding(tokenizer))
trainer.train()

## 5. Test on answers it has never seen
**Macro F1** counts the three verdicts equally, so it is the fairest single number.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

for name, df in tests.items():
    result = trainer.predict(Pairs(df))
    guess = result.predictions.argmax(-1)
    print("=== %s (%d pairs): accuracy %.1f%%, macro F1 %.3f" % (
        name, len(df), 100 * accuracy_score(result.label_ids, guess),
        f1_score(result.label_ids, guess, average="macro")))
    print(classification_report(result.label_ids, guess, labels=[0, 1, 2],
                                target_names=LABELS, zero_division=0))

## 6. Save and download

In [ ]:
trainer.save_model("prj022_model")
tokenizer.save_pretrained("prj022_model")
shutil.make_archive("prj022_model", "zip", "prj022_model")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("prj022_model.zip")
else:
    print("Done. Download prj022_model.zip from the Output panel.")